# 06 — Demo del pipeline end-to-end

Carga los modelos entrenados (pants y tops) y procesa un puñado de imágenes
de prueba, mostrando segmentación + paleta de colores + predicción
de atributos.


## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os
REPO_DIR = '/content/fashion-feature-extraction'
if not os.path.exists(REPO_DIR):
    SRC = '/content/drive/MyDrive/master_ia/fashion-extraction/code'
    import shutil
    shutil.copytree(SRC, REPO_DIR)
%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)
!pip install -q -r requirements.txt


## 2. Inicializar pipeline

In [ ]:
# El pipeline se construye con paths de Drive si correr en Colab
from src.data.colab import setup_colab
import yaml, tempfile, os

config = setup_colab('config/pipeline_config.yaml')

# Guardamos el config modificado para que FashionPipeline lo lea
patched_config_path = '/content/config_patched.yaml'
with open(patched_config_path, 'w') as f:
    yaml.safe_dump(config, f)

from src.pipeline import FashionPipeline
pipeline = FashionPipeline(patched_config_path)


## 3. Procesar imágenes de prueba

In [ ]:
# Tomamos 6 imagenes aleatorias del cache (3 pants + 3 tops)
import random, json
from pathlib import Path

random.seed(42)
pants_imgs = list(Path(config['paths']['images_pants']).glob('*.jpg'))[:3]
tops_imgs = list(Path(config['paths']['images_tops']).glob('*.jpg'))[:3]
sample = random.sample(pants_imgs + tops_imgs, min(6, len(pants_imgs) + len(tops_imgs)))

results = pipeline.process_batch(sample)
for r in results:
    print(json.dumps(r, indent=2, ensure_ascii=False)[:400], '\n---')


## 4. Visualización

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

n = len(results)
fig, axes = plt.subplots(1, n, figsize=(4 * n, 5))
if n == 1: axes = [axes]
for ax, r in zip(axes, results):
    img = Image.open(r['image_path'])
    ax.imshow(img); ax.axis('off')
    title = f"tipo: {r.get('garment_type', '?')}\n"
    attrs = r.get('attributes', {})
    for k, v in list(attrs.items())[:3]:
        title += f"{k}: {v.get('label', '?')} ({v.get('confidence', 0):.2f})\n"
    ax.set_title(title, fontsize=9)
plt.tight_layout(); plt.show()

## 5. Evaluación contra ground truth (opcional)

In [ ]:
# Si las imagenes vienen de los CSV, podemos comparar con ground truth
import pandas as pd

# Cargar CSVs
df_pants = pd.read_csv(config['data']['pants_csv'])
df_tops = pd.read_csv(config['data']['tops_csv'])

# Para cada result intentamos encontrar el id en los CSV
# (asume que las imagenes tienen el id en el nombre)
print('Para evaluacion completa, ejecuta:')
print('  python scripts/evaluate.py --predictions outputs/results.json \\')
print('         --csv data/preprocessed/pants_1.csv --dataset pants')
